# AMPS II · Week 01 — HW1
## Elasticity: rigid-body motions, locking, multigrid, equilibrium, and an LLM-generated hyperelastic script

**Due before Week 2 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q4 runs an LLM-generated script that is **meant** to be broken.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Rigid-body motions, Korn, compatibility of the pure-traction problem (theory) | 20 |
| 2 | Cantilever: slenderness and locking, multigrid with and without the near-nullspace | 40 |
| 3 | Reactions: force and moment equilibrium | 20 |
| 4 | Debugging an LLM-generated neo-Hookean load-stepping script | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model module from the lab (unchanged). Run it once.

In [ ]:
%%writefile /tmp/fnx/beam.py
"""Cantilever L x B x H, clamped at x = 0, total shear load P on x = L. Linear elasticity, CG + GAMG, reactions."""
import json, os
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh, la
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

E, nu = 1.0, 0.3
mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
eps = lambda u: ufl.sym(ufl.grad(u))
sigma = lambda u: 2 * mu * eps(u) + lmbda * ufl.tr(eps(u)) * ufl.Identity(3)

def timoshenko(P, L, B=1.0, H=1.0):
    """tip deflection of a Timoshenko cantilever (Cowper's shear coefficient)"""
    I = B * H**3 / 12; kap = 10 * (1 + nu) / (12 + 11 * nu)
    return P * L**3 / (3 * E * I) + P * L / (kap * mu * B * H)

def make_beam(nx, L=10.0, B=1.0, H=1.0, ny=None):
    """tetrahedral mesh nx x ny x ny (default ny = nx * H / L, at least 1); facet tags: 1 clamp (x = 0), 2 loaded end (x = L)"""
    ny = ny or max(1, round(nx * H / L))
    msh = mesh.create_box(comm, [[0, 0, 0], [L, B, H]], [nx, ny, ny], cell_type=mesh.CellType.tetrahedron)
    left = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[0], 0.0))
    right = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[0], L))
    fac = np.r_[left, right]; o = np.argsort(fac)
    ft = mesh.meshtags(msh, 2, fac[o], np.r_[np.full(len(left), 1), np.full(len(right), 2)][o].astype(np.int32))
    return msh, left, ufl.Measure("ds", domain=msh, subdomain_data=ft)

def rigid_body_modes(V):
    """the six rigid-body motions as a PETSc near-nullspace (API sheet §16b)"""
    x = V.tabulate_dof_coordinates()
    vecs = [la.vector(V.dofmap.index_map, V.dofmap.index_map_bs) for _ in range(6)]
    b = [w.array.reshape(-1, 3) for w in vecs]
    for i in range(3):
        b[i][:, i] = 1.0
    b[3][:, 0], b[3][:, 1] = -x[:, 1], x[:, 0]
    b[4][:, 0], b[4][:, 2] = x[:, 2], -x[:, 0]
    b[5][:, 2], b[5][:, 1] = x[:, 1], -x[:, 2]
    la.orthonormalize(vecs)
    return PETSc.NullSpace().create(vectors=[w.petsc_vec for w in vecs])

def solve_cantilever(nx, deg, P=1e-3, L=10.0, B=1.0, H=1.0, ny=None, nullspace=True, pc="gamg", rtol=1e-10):
    """returns (uh, info); info: dofs, CG iterations, converged reason, tip deflection (average u_z on x = L, positive down)"""
    msh, left, ds = make_beam(nx, L, B, H, ny)
    V = fem.functionspace(msh, ("Lagrange", deg, (3,)))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    t = fem.Constant(msh, np.array([0.0, 0.0, -P / (B * H)]))
    a = fem.form(ufl.inner(sigma(u), eps(v)) * ufl.dx); Lf = fem.form(ufl.dot(t, v) * ds(2))
    bc = fem.dirichletbc(np.zeros(3), fem.locate_dofs_topological(V, 2, left), V)
    A = assemble_matrix(a, bcs=[bc]); A.assemble()
    if nullspace:
        A.setNearNullSpace(rigid_body_modes(V))
    b = assemble_vector(Lf); apply_lifting(b, [a], [[bc]])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, [bc])
    ksp = PETSc.KSP().create(comm); ksp.setOperators(A); ksp.setType("cg"); ksp.getPC().setType(pc)
    ksp.setTolerances(rtol=rtol, max_it=5000)
    uh = fem.Function(V); ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
    tip = -comm.allreduce(fem.assemble_scalar(fem.form(uh[2] * ds(2))), op=MPI.SUM) / (B * H)
    info = dict(dofs=V.dofmap.index_map.size_global * 3, its=ksp.getIterationNumber(), reason=ksp.getConvergedReason(), tip=tip)
    return uh, dict(info, a=a, L=Lf, left=left, ds=ds, msh=msh)

def residual_on_clamp(uh, info):
    """r = K u - f with K and f assembled WITHOUT boundary conditions; returns (coordinates, r) of the clamped DOFs owned here"""
    V = uh.function_space
    A0 = assemble_matrix(info["a"]); A0.assemble()
    f0 = assemble_vector(info["L"]); f0.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    r = A0.createVecLeft(); A0.mult(uh.x.petsc_vec, r); r.axpy(-1.0, f0)
    dofs = fem.locate_dofs_topological(V, 2, info["left"]); dofs = dofs[dofs < V.dofmap.index_map.size_local]
    return V.tabulate_dof_coordinates()[dofs], r.array.reshape(-1, 3)[dofs]

def neo_hookean_cantilever(P, nsteps, nx=20, deg=2, L=10.0, B=1.0, H=1.0, linesearch="none"):
    """dead tip load P in nsteps equal steps; returns [(load, Newton its, reason, tip deflection)]"""
    from dolfinx.fem.petsc import NonlinearProblem
    msh, left, ds = make_beam(nx, L, B, H)
    V = fem.functionspace(msh, ("Lagrange", deg, (3,))); u = fem.Function(V); v = ufl.TestFunction(V)
    bc = fem.dirichletbc(np.zeros(3), fem.locate_dofs_topological(V, 2, left), V)
    F = ufl.Identity(3) + ufl.grad(u); C = F.T * F; J = ufl.det(F)
    psi = mu / 2 * (ufl.tr(C) - 3) - mu * ufl.ln(J) + lmbda / 2 * ufl.ln(J) ** 2
    t = fem.Constant(msh, np.zeros(3))
    R = ufl.derivative(psi * ufl.dx - ufl.dot(t, u) * ds(2), u, v)
    pr = NonlinearProblem(R, u, bcs=[bc], petsc_options_prefix=f"nh{nsteps}_{linesearch}_",
                          petsc_options={"snes_type": "newtonls", "snes_linesearch_type": linesearch, "snes_rtol": 1e-10,
                                         "snes_atol": 1e-12, "snes_max_it": 30, "ksp_type": "preonly", "pc_type": "lu",
                                         "pc_factor_mat_solver_type": "mumps"})
    out = []
    for k in range(1, nsteps + 1):
        t.value = np.array([0.0, 0.0, -k / nsteps * P / (B * H)])
        pr.solve(); s = pr.solver
        tip = -comm.allreduce(fem.assemble_scalar(fem.form(u[2] * ds(2))), op=MPI.SUM) / (B * H)
        out.append((k / nsteps * P, s.getIterationNumber(), s.getConvergedReason(), tip))
        if s.getConvergedReason() <= 0:
            break
    return out

## Q1 — Rigid-body motions and compatibility (20 pts)

**(a) (8)** Show that $\boldsymbol\varepsilon(\mathbf r)=\mathbf 0$ for $\mathbf r(\mathbf x)=\mathbf a+\boldsymbol\omega\times\mathbf x$. Conversely, show that
$\boldsymbol\varepsilon(\mathbf v)=\mathbf 0$ in a connected domain implies that $\mathbf v$ is of this form. *(Hint: $\partial_j\partial_k v_i$ can be written with
derivatives of $\varepsilon_{ij}$ only.)* How many independent rigid-body motions are there in 2D and in 3D?

**(b) (6)** For the pure-traction problem ($\Gamma_D=\emptyset$), show that a solution can exist only if $\ell(\mathbf r)=0$ for all rigid-body
motions $\mathbf r$, and interpret this condition physically.

**(c) (6)** In the lecture, P1 tetrahedra lock in bending. Give the mechanism in two or three sentences. Predict how the P1 tip-deflection ratio
changes (i) on a **fixed** mesh of $20\times2\times2$ elements when the beam is made more slender, (ii) when only the number of elements **along**
the beam is increased, (iii) when only the number **through the thickness** is increased. Q2(a) tests your prediction.

> *your answer*

## Q2 — Locking and multigrid (40 pts)

**(a) (20)** Tip deflection over Timoshenko ($H=B=1$, $P=10^{-3}$), P1 and P2, `-np 2`, in three studies:
(i) the fixed mesh $n_x=20$, `ny=2` for $L/H=5,10,20$; (ii) $L/H=10$, `ny=2`, $n_x=10,20,40,80$; (iii) $L/H=10$, $n_x=20$, `ny`$=1,2,4,8$.
Print the tables and compare with your prediction in Q1(c).

**(b) (20)** CG + GAMG iterations with and without the rigid-body near-nullspace for P2 on $n_x=10,20,40$ ($L/H=10$, default `ny`), and the
wall time of the solve (`time.perf_counter()` around `solve_cantilever`). What does the near-nullspace buy for P2 compared with P1 (lab §3)?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) — the three studies (i)-(iii), P1 and P2

> *your explanation (Q2a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) — GAMG iterations and solve time with/without the near-nullspace, P2, nx = 10, 20, 40

> *your explanation (Q2b)*

## Q3 — Force and moment equilibrium (20 pts)

The clamp carries the force $P$ **and** the moment $M=PL$ about the $y$-axis through the centre of the clamped face.
`residual_on_clamp(uh, info)` returns the coordinates and the residual vectors of the clamped DOFs (owned by each rank).

**(a) (12)** For P2, $n_x=10,20,40$: compute $R_z$ and $M_y=\sum_{\rm clamp}\big[(z-H/2)\,r_x-(x-0)\,r_z\big]$ from the residual, and the same moment from the
flux $\int_{\Gamma_D}\big[(z-H/2)\,(\boldsymbol\sigma\mathbf n)_x\big]ds$. Report $R_z/P$, $M_y/(PL)$ for both. *(Mind the sign convention: say which sign you expect.)*

**(b) (8)** Explain the results, and why "the reactions balance the load" is a useful **code** check even though the consistent value is exact by
construction.

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a)

> *your explanation (Q3)*

## Q4 — Debugging an LLM-generated hyperelastic script (20 pts)

An LLM was asked for "a neo-Hookean cantilever ($10\times1\times1$, P2, clamped at $x=0$) under a dead tip load $P=4\times10^{-3}$ in 8 load steps, printing
the Newton iterations and the tip deflection". The cell below is what it produced. It contains **four** defects.

**(a) (8)** Run it, and find and describe all four defects (what is wrong, how you found it).
**(b) (8)** Write the corrected script in the cell after it and run it.
**(c) (4)** Verify your corrected script: the small-load limit against `solve_cantilever` (linear, same mesh), and the converged reason of every step.

In [ ]:
%%fenicsx -np 2
# LLM-generated script (HW1 Q4) — it runs to the end, but it contains four defects
from mpi4py import MPI
import numpy as np, ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

comm = MPI.COMM_WORLD
L, B, H = 10.0, 1.0, 1.0
E, nu = 1.0, 0.3
mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
msh = mesh.create_box(comm, [[0, 0, 0], [L, B, H]], [20, 2, 2], cell_type=mesh.CellType.tetrahedron)
V = fem.functionspace(msh, ("Lagrange", 2, (3,)))
u = fem.Function(V); v = ufl.TestFunction(V)

left = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[0], 0.0))
right = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[0], L))
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 2, right, np.full(len(right), 2, dtype=np.int32)))

# clamp the left end
Vz = V.sub(2)
bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(Vz, 2, left), Vz)

# compressible neo-Hookean energy
F = ufl.Identity(3) + ufl.grad(u)
C = F.T * F
J = ufl.det(F)
Ic = ufl.tr(F)
psi = mu / 2 * (Ic - 3) - mu * ufl.ln(J) + lmbda / 2 * ufl.ln(J) ** 2

t = fem.Constant(msh, np.zeros(3))
R = ufl.derivative(psi * ufl.dx - ufl.dot(t, u) * ds(2), u, v)
problem = NonlinearProblem(R, u, bcs=[bc], petsc_options_prefix="hw1q4_",
                           petsc_options={"snes_type": "newtonls", "snes_linesearch_type": "none", "snes_rtol": 1e-10,
                                          "snes_atol": 1e-12, "snes_max_it": 30,
                                          "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})

P, nsteps = 4e-3, 8
for k in range(1, nsteps + 1):
    u.x.array[:] = 0.0                      # start each load step from rest
    t.value = np.array([0.0, 0.0, -k / nsteps * P / (B * H)])
    problem.solve()
    its = problem.solver.getIterationNumber()
    tip = comm.allreduce(np.abs(u.x.array).max(), op=MPI.MAX)
    if comm.rank == 0:
        print(f"step {k}: load {k / nsteps * P:.1e}  Newton its {its}  tip deflection {tip:.4f}")

> *your list of defects (Q4a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q4(b) — the corrected script

In [ ]:
%%fenicsx -np 2
# TODO: Q4(c) — small-load verification

> *your verification (Q4c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …